# Export ONNX: ResNet-18 → Raspberry Pi 5

Notebook này **không huấn luyện**. Nó lấy trọng số tốt nhất (`*_best.pt`) mà notebook huấn luyện ResNet-18 đã lưu, rồi:

1. Dựng lại **đúng** kiến trúc và tiền xử lý (các ô 0–4 được copy nguyên từ notebook huấn luyện).
2. Export sang **ONNX FP32** (batch 1, ảnh 224×224).
3. Lượng tử hóa **INT8** bằng ONNX Runtime: động (`int8_dyn`) và tĩnh có hiệu chỉnh (`int8_static`).
4. **Kiểm tra tương đương** trên tập test: PyTorch FP32 so với từng bản ONNX (tỉ lệ dự đoán trùng, sai khác logits, Acc/MCA/F1, độ chính xác từng lớp).
5. Đóng gói cho Pi: file `.onnx`, `meta.json` (tên lớp, kích thước ảnh, mean/std, kết quả kiểm tra) và một **tập con test** (ảnh + logits tham chiếu) để kiểm tra lại độ chính xác ngay trên Pi.

**Thiết lập Kaggle**
- *Settings → Accelerator*: GPU T4 x1 (chỉ dùng để tính logits tham chiếu PyTorch; ONNX Runtime chạy trên CPU).
- *Settings → Internet*: **On**.
- *Add Input*: **checkpoint đã huấn luyện**: dataset `fer-checkpoints` gồm các thư mục `<MODEL_NAME>_sqrt/` (hoặc Output của notebook huấn luyện ResNet-18). Ô **5** tự tìm thư mục đó trong `/kaggle/input`, kể cả khi Kaggle đã **tự giải nén file `.pt` thành thư mục** (code tự đóng gói lại).
- Mặc định chỉ export mô hình **FERPlus** (ô **1b**). Chỉ cần Add Input **RAF-DB** nếu thêm `"rafdb"` vào `EXPORT_DATASETS`.
- **FER2013 không bắt buộc Add Input**: nếu không thấy `fer2013.csv` trong `/kaggle/input`, ô **3** tự tải từ Kaggle bằng `kagglehub` (dataset `deadskull7/fer2013`, dự phòng `ashishpatel26/facial-expression-recognitionferchallenge`); số dòng được kiểm tra đủ 35.887 ảnh như lúc huấn luyện.

**Cách dùng**: kiểm tra ô **1. Cấu hình** giống hệt lúc huấn luyện (cùng `MODEL_NAME`, `SAMPLING`), chọn dataset ở ô **1b** → *Save Version → Save & Run All (Commit)*. Kết quả: các file **`.ferpack`** trong `/kaggle/working/packs/` (tải ở tab *Output*) — đây là thứ duy nhất cần mang đi.

## 0. Kiểm tra input và tải nhãn FERPlus
In ra đường dẫn thật của các dataset đã Add Input (dùng để điền ô Cấu hình nếu tự tìm không thành công).

In [ ]:
!ls /kaggle/input
!find /kaggle/input -maxdepth 4 -type d | head -40
!find /kaggle/input -maxdepth 5 -name "*.csv" | head -20

# Nhãn FERPlus của Microsoft (cần bật Internet)
!wget -q -O /kaggle/working/fer2013new.csv https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv
!head -2 /kaggle/working/fer2013new.csv

## 1. Cấu hình

In [ ]:
from pathlib import Path

# ---------------- Kiến trúc ----------------
MODEL_NAME = "resnet18"                      # "resnet18" hoặc "resnet50"
TIMM_NAMES = {"resnet18": "resnet18.a1_in1k", "resnet50": "resnet50.a1_in1k"}
LR_BY_MODEL = {"resnet18": 3e-4, "resnet50": 2e-4}

# ---------------- Thực nghiệm ----------------
DATASETS_TO_RUN = ["rafdb", "ferplus"]   # bỏ bớt nếu muốn chạy riêng từng dataset
SEEDS = [0]                              # bài báo: [0, 1, 2] để báo cáo mean ± std

# ---------------- Siêu tham số chung (giống nhau cho mọi kiến trúc sau này) ----------------
IMG_SIZE = 224
BATCH_SIZE = 64
LR = LR_BY_MODEL[MODEL_NAME]
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 2
LABEL_SMOOTHING = 0.1
# Xử lý mất cân bằng lớp: lấy mẫu lại theo CĂN BẬC HAI tần suất lớp (chỉ áp dụng cho tập train).
# Lớp hiếm được lấy lặp lại nhiều hơn, mỗi lần lại qua augmentation ngẫu nhiên khác nhau.
SAMPLING = "sqrt"
EARLY_STOP_PATIENCE = 10      # dừng nếu val accuracy không tăng sau N epoch (0 = tắt)
NUM_WORKERS = 4
USE_AMP = True                # mixed precision fp16

# ---------------- Dữ liệu: SỬA ĐƯỜNG DẪN CHO KHỚP VỚI INPUT CỦA BẠN ----------------
# type="folder": root/<train_dir>/<thư mục lớp>/*.jpg  (thư mục train/test có thể nằm sâu bên trong root, notebook tự tìm)
# class_dirs: tên thư mục của từng lớp, CÙNG THỨ TỰ với classes
DATASETS = {
    "rafdb": dict(
        type="folder",
        root="/kaggle/input/raf-db-dataset",
        train_dir="train", test_dir="test",
        class_dirs=["1", "2", "3", "4", "5", "6", "7"],
        classes=["surprise", "fear", "disgust", "happiness", "sadness", "anger", "neutral"],
        val_ratio=0.1,            # RAF-DB không có tập val -> tách 10% train (cố định)
        epochs=40,
    ),
    # FERPlus gốc = ảnh của fer2013.csv + nhãn mới của Microsoft (fer2013new.csv).
    # Nhãn = cảm xúc nhiều phiếu nhất trong 8 cảm xúc; bỏ ảnh mà phiếu 'unknown'/'NF' >= phiếu cao nhất.
    # Usage: Training -> train, PublicTest -> val, PrivateTest -> test (phân bố lớp tự nhiên).
    "ferplus": dict(
        type="ferplus_csv",
        fer2013_csv="auto",                                  # "auto" = tự tìm fer2013.csv trong /kaggle/input
        ferplus_csv="/kaggle/working/fer2013new.csv",        # do ô 0 tải về
        epochs=30,
    ),
}

OUT_DIR = Path("/kaggle/working/results") / f"{MODEL_NAME}_{SAMPLING}"   # thư mục riêng, không lẫn với kết quả cũ
SPLIT_SEED = 2026                          # cố định cách tách val cho mọi model -> so sánh công bằng
INPUT_ROOT = Path("/kaggle/input")         # nơi tự tìm dữ liệu khi đường dẫn khai báo không tồn tại
CACHE_DIR = Path("/tmp/fer_cache")         # ảnh FERPlus 48x48 giải nén từ CSV (không lưu vào output)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Model:", MODEL_NAME, "| lr:", LR, "| output:", OUT_DIR)

### 1b. Dataset cần export

In [ ]:
EXPORT_DATASETS = ["ferplus"]          # thêm "rafdb" nếu cần (khi đó phải Add Input dataset RAF-DB)
DATASETS_TO_RUN = [d for d in DATASETS_TO_RUN if d in EXPORT_DATASETS]
assert MODEL_NAME == "resnet18", f"Notebook này dành cho resnet18, nhưng MODEL_NAME = {MODEL_NAME}"
print("export:", MODEL_NAME, DATASETS_TO_RUN)

## 2. Thư viện

In [ ]:
import json, math, os, random, time, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
import timm

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| timm", timm.__version__, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Đọc dữ liệu
Mỗi dataset được chuyển về 3 bảng `train / val / test` gồm cột `path, label`.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def find_dir(root, name, max_depth=4):
    """Tìm thư mục tên `name` (không phân biệt hoa thường) bên dưới root, ưu tiên nông nhất."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Không tồn tại: {root}")
    level = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in level:
            if d.name.lower() == name.lower() and d != root:
                return d
            try:
                subs = [p for p in d.iterdir() if p.is_dir()]
            except PermissionError:
                continue
            if len(subs) < 200:          # không lặn vào thư mục chứa hàng nghìn thư mục con
                nxt += subs
        level = nxt
    raise FileNotFoundError(f"Không tìm thấy thư mục '{name}' trong {root}")


def list_images(d):
    out = []
    for dp, _, files in os.walk(d):
        out += [os.path.join(dp, f) for f in files if os.path.splitext(f)[1].lower() in IMG_EXTS]
    return sorted(out)


def load_folder_split(root, split_dir, class_dirs):
    base = find_dir(root, split_dir)
    present = {p.name.lower(): p for p in base.iterdir() if p.is_dir()}
    rows = []
    for label, cd in enumerate(class_dirs):
        if cd.lower() not in present:
            raise FileNotFoundError(f"Thiếu thư mục lớp '{cd}' trong {base}. Có: {sorted(present)}")
        rows += [(p, label) for p in list_images(present[cd.lower()])]
    return pd.DataFrame(rows, columns=["path", "label"])


def auto_find_folder_root(train_dir, test_dir, class_dirs, search_root, max_depth=5):
    """Tìm thư mục chứa cả <train_dir>/ và <test_dir>/, trong <train_dir>/ có đủ các thư mục lớp."""
    want = {c.lower() for c in class_dirs}
    for dp, dirs, _ in os.walk(search_root):
        if len(Path(dp).relative_to(search_root).parts) >= max_depth:
            dirs[:] = []
            continue
        low = {d.lower(): d for d in dirs}
        if train_dir.lower() in low and test_dir.lower() in low:
            sub = {s.lower() for s in os.listdir(os.path.join(dp, low[train_dir.lower()]))}
            if want <= sub:
                return Path(dp)
    return None


# Nguồn tự tải fer2013.csv khi notebook chưa Add Input dataset FER2013 (thử lần lượt; số dòng được kiểm tra = 35887)
FER2013_KAGGLE = ["deadskull7/fer2013", "ashishpatel26/facial-expression-recognitionferchallenge"]
FERPLUS_LABELS_URL = "https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv"


def _walk_find(root, names):
    for dp, _, files in os.walk(root, followlinks=True):     # followlinks: Kaggle có thể gắn input qua liên kết tắt
        for n in names:
            if n in files:
                return Path(dp) / n
    return None


def find_csv(path, names, search_root):
    """Trả về path nếu tồn tại; nếu không thì tìm dưới search_root rồi /kaggle/input;
    nếu vẫn không có: fer2013.csv được tự tải từ Kaggle (kagglehub), fer2013new.csv tự tải từ GitHub của Microsoft."""
    if path != "auto" and Path(path).exists():
        return Path(path)
    for root in dict.fromkeys([Path(search_root), INPUT_ROOT]):
        hit = _walk_find(root, names)
        if hit:
            return hit
    if "fer2013new.csv" in names:
        import urllib.request
        dst = Path("/kaggle/working/fer2013new.csv")
        print(f"  tự tải nhãn FERPlus -> {dst}")
        urllib.request.urlretrieve(FERPLUS_LABELS_URL, dst)
        return dst
    if "fer2013.csv" in names:
        try:
            import kagglehub
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
            import kagglehub
        for handle in FER2013_KAGGLE:
            try:
                print(f"  chưa có fer2013.csv trong input -> tự tải dataset Kaggle '{handle}' (cần bật Internet) ...")
                hit = _walk_find(kagglehub.dataset_download(handle), names)
                if hit:
                    return hit
                print(f"  '{handle}' không chứa {names}")
            except Exception as e:
                print(f"  tải '{handle}' lỗi: {type(e).__name__}: {str(e)[:200]}")
    raise FileNotFoundError(f"Không tìm thấy {names} (đường dẫn khai báo: {path}) và không tự tải được. "
                            f"Hãy bật Internet hoặc Add Input dataset FER2013 có file fer2013.csv.")


FERPLUS_CLASSES = ["neutral", "happiness", "surprise", "sadness", "anger", "disgust", "fear", "contempt"]


def load_ferplus(fer2013_csv, ferplus_csv, cache_dir):
    fer_path = find_csv(fer2013_csv, ["fer2013.csv", "icml_face_data.csv"], INPUT_ROOT)
    new_path = find_csv(ferplus_csv, ["fer2013new.csv"], INPUT_ROOT if ferplus_csv == "auto" else Path("/kaggle/working"))
    print(f"  ảnh : {fer_path}\n  nhãn: {new_path}")
    fer = pd.read_csv(fer_path)
    fer.columns = [c.strip() for c in fer.columns]       # icml_face_data.csv có khoảng trắng trong tên cột
    new = pd.read_csv(new_path)
    if not {"pixels", "Usage"} <= set(fer.columns):
        raise ValueError(f"{fer_path.name} thiếu cột 'pixels'/'Usage' (cột hiện có: {list(fer.columns)}). "
                         "Bạn đang dùng bản FER2013 dạng thư mục ảnh? Cần bản có file CSV.")
    if len(fer) != len(new):
        raise ValueError(f"Số dòng không khớp: {fer_path.name} có {len(fer)}, fer2013new.csv có {len(new)} (cần 35887).")
    cols = {c.lower(): c for c in new.columns}
    votes = new[[cols[c] for c in FERPLUS_CLASSES]].to_numpy()
    junk = np.maximum(new[cols["unknown"]].to_numpy(), new[cols["nf"]].to_numpy())
    usage = {"Training": "train", "PublicTest": "val", "PrivateTest": "test"}
    img_dir = Path(cache_dir) / "ferplus"
    img_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, (px, use) in enumerate(zip(fer["pixels"], fer["Usage"].astype(str).str.strip())):
        top = votes[i].max()
        if top <= 0 or junk[i] >= top or use not in usage:
            continue
        f = img_dir / f"{i:05d}.png"
        if not f.exists():
            Image.fromarray(np.asarray(px.split(), dtype=np.uint8).reshape(48, 48)).save(f)
        rows.append((str(f), int(votes[i].argmax()), usage[use]))
    df = pd.DataFrame(rows, columns=["path", "label", "split"])
    parts = [df[df.split == s][["path", "label"]].reset_index(drop=True) for s in ("train", "val", "test")]
    return (*parts, list(FERPLUS_CLASSES))


def load_dataset(name):
    c = DATASETS[name]
    if c["type"] == "ferplus_csv":
        return load_ferplus(c["fer2013_csv"], c["ferplus_csv"], CACHE_DIR)
    if not Path(c["root"]).exists():
        found = auto_find_folder_root(c["train_dir"], c["test_dir"], c["class_dirs"], INPUT_ROOT)
        if found is None:
            raise FileNotFoundError(f"[{name}] '{c['root']}' không tồn tại và không tự tìm được thư mục chứa "
                                    f"'{c['train_dir']}/' + '{c['test_dir']}/'. Xem kết quả ô 0 và sửa 'root'.")
        print(f"  [{name}] '{c['root']}' không tồn tại -> dùng {found}")
        c["root"] = str(found)
    train = load_folder_split(c["root"], c["train_dir"], c["class_dirs"])
    test = load_folder_split(c["root"], c["test_dir"], c["class_dirs"])
    # RAF-DB không có val -> tách phân tầng từ train, cố định SPLIT_SEED
    train, val = train_test_split(train, test_size=c.get("val_ratio", 0.1), stratify=train["label"],
                                  random_state=SPLIT_SEED)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True), c["classes"]


DATA = {}
for name in DATASETS_TO_RUN:
    tr, va, te, classes = load_dataset(name)
    DATA[name] = (tr, va, te, classes)
    counts = pd.DataFrame({s: d["label"].value_counts().sort_index() for s, d in (("train", tr), ("val", va), ("test", te))})
    counts.index = [classes[i] for i in counts.index]
    print(f"\n===== {name}: train {len(tr)} | val {len(va)} | test {len(te)}")
    display(counts.fillna(0).astype(int).T)

## 4. Dataset, augmentation, model

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.RandomApply([T.ColorJitter(0.2, 0.2, 0.2)], p=0.5),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.25)),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])


class FERDataset(Dataset):
    def __init__(self, df, tf):
        self.paths, self.labels, self.tf = df["path"].tolist(), df["label"].to_numpy(), tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")      # ảnh xám FERPlus -> 3 kênh
        return self.tf(img), int(self.labels[i])


def sqrt_sample_weights(labels):
    """Trọng số từng ảnh = 1/sqrt(n_lớp)  ->  xác suất lấy một lớp tỉ lệ với sqrt(n_lớp)."""
    counts = np.bincount(labels)
    return 1.0 / np.sqrt(counts[labels])


def expected_class_share(labels):
    """Tỉ lệ (%) mỗi lớp trong một epoch: trước và sau khi lấy mẫu lại."""
    counts = np.bincount(labels)
    return 100 * counts / counts.sum(), 100 * np.sqrt(counts) / np.sqrt(counts).sum()


def make_loaders(tr, va, te, seed):
    g = torch.Generator().manual_seed(seed)
    ds_tr = FERDataset(tr, train_tf)
    # Lấy có hoàn lại, số mẫu mỗi epoch = kích thước tập train (thời gian 1 epoch không đổi)
    w = torch.as_tensor(sqrt_sample_weights(ds_tr.labels), dtype=torch.double)
    sampler = WeightedRandomSampler(w, num_samples=len(ds_tr), replacement=True, generator=g)
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
    return (DataLoader(ds_tr, BATCH_SIZE, sampler=sampler, drop_last=True, **kw),
            DataLoader(FERDataset(va, eval_tf), BATCH_SIZE * 2, shuffle=False, **kw),
            DataLoader(FERDataset(te, eval_tf), BATCH_SIZE * 2, shuffle=False, **kw))


def build_model(num_classes):
    return timm.create_model(TIMM_NAMES[MODEL_NAME], pretrained=True, num_classes=num_classes)


def model_cost(model):
    """Số tham số (M), GMACs/ảnh (= FLOPs/2, tương đương 'FLOPs' trong đa số bài FER), FPS (batch 1)."""
    params = sum(p.numel() for p in model.parameters()) / 1e6
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            model(x)
        gmacs = fc.get_total_flops() / 2e9
    except Exception:
        gmacs = float("nan")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
        for _ in range(20):
            model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(100):
            model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return params, gmacs, 100 / (time.perf_counter() - t0)


m = build_model(7).to(DEVICE)
print("%s: %.2fM params, %.2f GMACs, %.0f FPS" % (MODEL_NAME, *model_cost(m)))
del m

## 5. Cấu hình export và tìm checkpoint
Checkpoint được tìm theo tên thư mục `<MODEL_NAME>_<SAMPLING>` (ví dụ `resnet18_sqrt`) trong `/kaggle/input`. Nếu tự tìm không được, điền đường dẫn vào `CKPT_DIR` (xem kết quả ô 0).

In [ ]:
import gc, inspect, shutil, subprocess, sys, zipfile
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript"], check=False)
import onnx
import onnxruntime as ort
from torch.nn.utils import parametrize as _parametrize

# ---------------- Export ----------------
EXPORT_SEED = 0              # seed của checkpoint cần export (phải có trong SEEDS lúc huấn luyện)
CKPT_DIR = "auto"            # "auto" = tự tìm thư mục <MODEL_NAME>_<SAMPLING> trong /kaggle/input
EXPORT_DIR = Path("/kaggle/working/export") / MODEL_NAME
OPSET = 17
QUANT_MODES = ["int8_dyn", "int8_static"]   # bỏ bớt nếu không cần
N_CALIB = 200                # số ảnh train (không augmentation) để hiệu chỉnh INT8 tĩnh
N_PI_SUBSET = 500            # số ảnh test (phân tầng theo lớp) mang sang Pi để kiểm tra lại độ chính xác
PARITY_MAX_IMAGES = None     # None = toàn bộ tập test (dùng cho bài báo); đặt 300 để chạy thử nhanh
ORT_THREADS = os.cpu_count()
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Tắt các đường tắt có thể làm lệch số học giữa PyTorch và ONNX
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
try:
    torch.backends.mha.set_fastpath_enabled(False)      # nn.MultiheadAttention: dùng đường export được
except Exception:
    pass


def find_ckpt_dir():
    if CKPT_DIR != "auto":
        return Path(CKPT_DIR)
    want = f"{MODEL_NAME}_{SAMPLING}"
    hits = sorted({Path(dp) for dp, dirs, files in os.walk(INPUT_ROOT, followlinks=True)   # Kaggle gắn input qua liên kết tắt
                   if Path(dp).name == want and (any(f.endswith("_best.pt") for f in files)
                                                 or any(d.endswith("_best") for d in dirs))})   # .pt bị Kaggle giải nén
    if not hits:
        raise FileNotFoundError(
            f"Không tìm thấy thư mục '{want}' chứa *_best.pt trong {INPUT_ROOT}. Hãy Add Input Output của notebook "
            f"huấn luyện (phiên bản đã Save & Run All), hoặc điền CKPT_DIR. Thư mục input hiện có: "
            f"{sorted(p.name for p in INPUT_ROOT.iterdir())}")
    if len(hits) > 1:
        print("Cảnh báo: thấy nhiều thư mục checkpoint, dùng thư mục đầu tiên:", *hits, sep="\n  ")
    return hits[0]


CKPT = find_ckpt_dir()
print("checkpoint:", CKPT)
for p in sorted(CKPT.iterdir()):
    if p.is_file():
        print(f"  {p.name:40s} {p.stat().st_size / 1e6:8.1f} MB")
    else:
        print(f"  {p.name + '/':40s} (thư mục: file .pt đã bị giải nén, sẽ được đóng gói lại)")
REF = pd.read_csv(CKPT / "results.csv") if (CKPT / "results.csv").exists() else None
if REF is not None:
    display(REF[["model", "dataset", "seed", "test_acc", "test_mca", "test_f1", "best_epoch"]])
print("torch", torch.__version__, "| onnx", onnx.__version__, "| onnxruntime", ort.__version__, "| CPU threads", ORT_THREADS)

## 6. Hàm export, lượng tử hóa và kiểm tra
- **Export**: thử bộ export TorchScript (`dynamo=False`, opset 17) trước, nếu lỗi thì thử bộ export mới (`dynamo=True`, opset ≥ 18). Đầu vào cố định `1×3×IMG_SIZE×IMG_SIZE`, tên `input`; đầu ra `logits`.
- **INT8 động** (`quantize_dynamic`): trọng số INT8, activation lượng tử hóa lúc chạy; không cần dữ liệu hiệu chỉnh.
- **INT8 tĩnh** (`quantize_static`, định dạng QDQ, trọng số theo kênh, MinMax): hiệu chỉnh bằng `N_CALIB` ảnh train qua đúng tiền xử lý đánh giá.
- **Tương đương**: logits ONNX Runtime (CPU, batch 1) so với PyTorch FP32 trên cùng ảnh; báo tỉ lệ dự đoán trùng, sai khác logits lớn nhất và Acc/MCA/macro-F1/độ chính xác từng lớp.

In [ ]:
from onnxruntime.quantization import (CalibrationDataReader, CalibrationMethod, QuantFormat, QuantType,
                                      quantize_dynamic, quantize_static)
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
except Exception:
    quant_pre_process = None


def make_model(classes):
    return build_model(len(classes))


def resolve_ckpt(name, seed):
    """Trả về file .pt. Nếu Kaggle đã giải nén .pt thành thư mục <name>_s<seed>_best/, đóng gói lại vào /kaggle/working."""
    import zipfile
    stem = f"{name}_s{seed}_best"
    if (CKPT / f"{stem}.pt").is_file():
        return CKPT / f"{stem}.pt"
    src = CKPT / stem
    pkl = None
    if src.is_dir():
        for dp, _, fs in os.walk(src, followlinks=True):
            if "data.pkl" in fs:
                pkl = Path(dp) / "data.pkl"
                break
    if pkl is None:
        raise FileNotFoundError(f"Không thấy {stem}.pt hoặc thư mục {stem}/ chứa data.pkl trong {CKPT}")
    dst = Path("/kaggle/working/ckpt_fixed") / MODEL_NAME / f"{stem}.pt"
    dst.parent.mkdir(parents=True, exist_ok=True)
    root = pkl.parent
    with zipfile.ZipFile(dst, "w", zipfile.ZIP_STORED) as z:
        for dp, _, fs in os.walk(root, followlinks=True):
            for f in sorted(fs):
                fp = Path(dp) / f
                z.write(fp, f"{stem}/{fp.relative_to(root).as_posix()}")
    print(f"  {stem}/ là file .pt đã bị giải nén -> đóng gói lại thành {dst}")
    return dst


def load_finetuned(name, seed):
    classes = DATA[name][3]
    path = resolve_ckpt(name, seed)
    model = make_model(classes)
    sd = torch.load(path, map_location="cpu")
    try:
        model.load_state_dict(sd, strict=True)
    except RuntimeError as e:
        raise RuntimeError(f"Checkpoint {path.name} không khớp kiến trúc. Kiểm tra ô Cấu hình có giống lúc huấn luyện "
                           f"không (MODEL_NAME, PEFT_METHOD, phiên bản thư viện).\n{e}") from e
    merged = 0                       # gộp mọi parametrization (LoRA của CLIP) vào trọng số thường
    for mod in list(model.modules()):
        if _parametrize.is_parametrized(mod):
            for pname in list(mod.parametrizations.keys()):
                _parametrize.remove_parametrizations(mod, pname, leave_parametrized=True)
                merged += 1
    model = model.float().eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    return model, classes, path, merged, n_params


def cls_metrics(y, p, k):
    cm = np.zeros((k, k), dtype=np.int64)
    np.add.at(cm, (y, p), 1)
    tp = np.diag(cm).astype(float)
    rec = tp / np.maximum(cm.sum(1), 1)
    prec = tp / np.maximum(cm.sum(0), 1)
    f1 = np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-12), 0.0)
    return {"acc": 100 * float((y == p).mean()), "mca": 100 * float(rec.mean()), "f1": 100 * float(f1.mean()),
            "per_class": [round(100 * float(r), 2) for r in rec]}


def loader_of(df, bs=32):
    return DataLoader(FERDataset(df, eval_tf), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)


@torch.no_grad()
def torch_logits(model, df):
    model.to(DEVICE)
    out = [model(x.to(DEVICE)).float().cpu().numpy() for x, _ in loader_of(df, BATCH_SIZE)]
    model.cpu()
    torch.cuda.empty_cache()
    return np.concatenate(out)


def ort_session(path, threads=ORT_THREADS):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])


def ort_logits(path, df):
    sess = ort_session(path)
    iname = sess.get_inputs()[0].name
    out = []
    for x, _ in loader_of(df):
        for xi in x.numpy():                                   # chạy từng ảnh: batch 1 như trên robot
            out.append(sess.run(None, {iname: xi[None]})[0][0])
    return np.stack(out).astype(np.float32)


def export_onnx(model, path):
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    has_dynamo = "dynamo" in inspect.signature(torch.onnx.export).parameters
    errors = []
    for use_dynamo in ([False, True] if has_dynamo else [False]):
        try:
            kw = dict(input_names=["input"], output_names=["logits"],
                      opset_version=max(OPSET, 18) if use_dynamo else OPSET)
            if has_dynamo:
                kw["dynamo"] = use_dynamo
            if not use_dynamo:
                kw["do_constant_folding"] = True
            with torch.no_grad():
                torch.onnx.export(model, (x,), str(path), **kw)
            m = onnx.load(str(path))                            # gộp external data (nếu có) vào 1 file
            onnx.checker.check_model(m)
            onnx.save(m, str(path))
            for extra in path.parent.glob(path.name + ".data"):
                extra.unlink()
            return ("dynamo" if use_dynamo else "torchscript"), int(m.opset_import[0].version)
        except Exception as e:
            errors.append(f"{'dynamo' if use_dynamo else 'torchscript'}: {type(e).__name__}: {str(e)[:800]}")
            print("  export lỗi ->", errors[-1][:300])
    raise RuntimeError("Không export được ONNX:\n" + "\n".join(errors))


class CalibReader(CalibrationDataReader):
    def __init__(self, df, iname):
        self.items = iter([{iname: x.numpy()[None]} for x, _ in FERDataset(df, eval_tf)])

    def get_next(self):
        return next(self.items, None)


def calib_subset(name):
    tr = DATA[name][0]
    return tr.sample(n=min(N_CALIB, len(tr)), random_state=0).reset_index(drop=True)


def preprocess_for_quant(fp32):
    pre = fp32.with_name("_prep.onnx")
    if quant_pre_process is None:
        return fp32
    for kw in ({}, {"skip_symbolic_shape": True}, {"skip_symbolic_shape": True, "skip_optimization": True}):
        try:
            quant_pre_process(str(fp32), str(pre), **kw)
            return pre
        except Exception as e:
            print(f"  quant_pre_process{kw} lỗi: {type(e).__name__}: {str(e)[:200]}")
    return fp32


def quantize(fp32, pre, mode, name):
    out = fp32.with_name(f"model_{mode}.onnx")
    if mode == "int8_dyn":
        quantize_dynamic(str(pre), str(out), weight_type=QuantType.QInt8)
    elif mode == "int8_static":
        iname = ort_session(fp32, 1).get_inputs()[0].name
        quantize_static(str(pre), str(out), CalibReader(calib_subset(name), iname),
                        quant_format=QuantFormat.QDQ, per_channel=True,
                        activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8,
                        calibrate_method=CalibrationMethod.MinMax,
                        extra_options={"CalibMaxIntermediateOutputs": 32})
    else:
        raise ValueError(mode)
    onnx.checker.check_model(str(out))
    return out


def stratified(df, n, seed=0):
    if n >= len(df):
        return df.reset_index(drop=True)
    frac = n / len(df)
    parts = [d.sample(n=max(1, round(len(d) * frac)), random_state=seed) for _, d in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

## 7. Chạy export cho từng dataset
Mỗi dataset tạo một thư mục `export/<MODEL_NAME>/<dataset>/` gồm:
`model_fp32.onnx`, `model_int8_dyn.onnx`, `model_int8_static.onnx`, `meta.json`, `pi_subset/` (ảnh + `labels.csv` + `ref_logits.npz`).

In [ ]:
VERSIONS = {"torch": torch.__version__, "onnx": onnx.__version__, "onnxruntime": ort.__version__, "python": sys.version.split()[0]}
SUMMARY = []

for name in DATASETS_TO_RUN:
    print(f"\n########## {MODEL_NAME} | {name} ##########")
    out_dir = EXPORT_DIR / name
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    model, classes, ckpt, merged, n_params = load_finetuned(name, EXPORT_SEED)
    k = len(classes)
    print(f"  nạp {ckpt.name}: {n_params:.2f}M tham số" + (f", đã gộp {merged} LoRA" if merged else ""))

    te = DATA[name][2]
    te_eval = te if PARITY_MAX_IMAGES is None else stratified(te, PARITY_MAX_IMAGES)
    y = te_eval["label"].to_numpy()
    pi_df = stratified(te, N_PI_SUBSET)

    ref = torch_logits(model, te_eval)                          # PyTorch FP32 (không AMP)
    ref_pi = torch_logits(model, pi_df)
    m_torch = cls_metrics(y, ref.argmax(1), k)
    print(f"  PyTorch FP32  acc {m_torch['acc']:.2f} | mca {m_torch['mca']:.2f} | f1 {m_torch['f1']:.2f}  ({len(y)} ảnh test)")

    files, errors = {}, {}
    fp32 = out_dir / "model_fp32.onnx"
    exporter, opset = export_onnx(model, fp32)
    files["fp32"] = fp32
    print(f"  export: {exporter}, opset {opset}, {fp32.stat().st_size / 1e6:.1f} MB")
    del model
    gc.collect()

    if QUANT_MODES:
        pre = preprocess_for_quant(fp32)
        for mode in QUANT_MODES:
            t0 = time.time()
            try:
                files[mode] = quantize(fp32, pre, mode, name)
                print(f"  {mode}: {files[mode].stat().st_size / 1e6:.1f} MB ({time.time() - t0:.0f}s)")
            except Exception as e:
                errors[mode] = f"{type(e).__name__}: {str(e)[:500]}"
                print(f"  {mode} LỖI: {errors[mode][:300]}")
        if pre != fp32 and pre.exists():
            pre.unlink()

    # ---- kiểm tra tương đương trên tập test + logits tham chiếu cho tập con của Pi
    results, ref_npz = {}, {"torch_fp32": ref_pi}
    for variant, path in files.items():
        t0 = time.time()
        lg = ort_logits(path, te_eval)
        m = cls_metrics(y, lg.argmax(1), k)
        m.update(agree_torch=100 * float((lg.argmax(1) == ref.argmax(1)).mean()),
                 max_abs_logit_diff=float(np.abs(lg - ref).max()),
                 size_mb=round(path.stat().st_size / 1e6, 2), file=path.name)
        results[variant] = m
        ref_npz[variant] = ort_logits(path, pi_df)
        print(f"  ONNX {variant:12s} acc {m['acc']:.2f} | mca {m['mca']:.2f} | f1 {m['f1']:.2f} | "
              f"trùng PyTorch {m['agree_torch']:.2f}% | max|Δlogit| {m['max_abs_logit_diff']:.4f}  ({time.time() - t0:.0f}s)")
        SUMMARY.append({"model": MODEL_NAME, "dataset": name, "variant": variant, "n_test": len(y),
                        **{c: round(m[c], 2) for c in ("acc", "mca", "f1", "agree_torch")},
                        "acc_drop_vs_torch": round(m_torch["acc"] - m["acc"], 2),
                        "f1_drop_vs_torch": round(m_torch["f1"] - m["f1"], 2),
                        "max_abs_logit_diff": round(m["max_abs_logit_diff"], 4), "size_mb": m["size_mb"],
                        "per_class": json.dumps(dict(zip(classes, m["per_class"])))})

    # ---- tập con test cho Pi
    sub = out_dir / "pi_subset"
    (sub / "images").mkdir(parents=True)
    rows = []
    for i, r in pi_df.iterrows():
        dst = f"{i:04d}{Path(r['path']).suffix.lower()}"
        shutil.copy(r["path"], sub / "images" / dst)
        rows.append({"file": dst, "label": int(r["label"]), "source": Path(r["path"]).name})
    pd.DataFrame(rows).to_csv(sub / "labels.csv", index=False)
    np.savez_compressed(sub / "ref_logits.npz", **ref_npz)

    ref_row = None
    if REF is not None:
        rr = REF[(REF.dataset == name) & (REF.seed == EXPORT_SEED)]
        if len(rr):
            ref_row = {c: float(rr.iloc[0][c]) for c in ("test_acc", "test_mca", "test_f1")}
    meta = {
        "model": MODEL_NAME, "dataset": name, "seed": EXPORT_SEED, "classes": list(classes), "num_classes": k,
        "img_size": IMG_SIZE, "mean": list(MEAN), "std": list(STD), "color": "RGB", "layout": "NCHW float32",
        "grayscale": name == "ferplus", "train_res": 48 if name == "ferplus" else None,   # camera: xám -> 48x48 -> IMG_SIZE
        "resize": "PIL Image.resize((S, S), BILINEAR) rồi /255, (x-mean)/std (giống torchvision Resize + ToTensor + Normalize)",
        "input_name": "input", "output_name": "logits", "batch": 1, "exporter": exporter, "opset": opset,
        "params_M": round(n_params, 2), "checkpoint": str(ckpt), "merged_parametrizations": merged,
        "files": {v: p.name for v, p in files.items()}, "quant_errors": errors,
        "test_eval": {"n": int(len(y)), "torch_fp32": m_torch, **results},
        "results_csv_ref": ref_row, "pi_subset": {"n": len(pi_df), "dir": "pi_subset"},
        "quant": {"int8_dyn": "quantize_dynamic, weight QInt8, per-tensor",
                  "int8_static": f"quantize_static QDQ, per-channel, act QUInt8 / weight QInt8, MinMax, {N_CALIB} ảnh train"},
        "versions": VERSIONS,
    }
    (out_dir / "meta.json").write_text(json.dumps(meta, indent=1, ensure_ascii=False), encoding="utf-8")
    print("  đã lưu", out_dir)

## 8. Tổng hợp và đóng gói
`acc_drop_vs_torch` = độ chính xác PyTorch FP32 trừ độ chính xác của bản ONNX. Bản **FP32** phải có `agree_torch` ≥ 99,5% và `acc_drop` ≈ 0; nếu không, export có vấn đề và **không dùng số đo trên Pi** của mô hình đó. Với INT8, so sánh thêm cột `per_class` để xem lớp thiểu số (fear, disgust, contempt) có bị ảnh hưởng nhiều hơn không.

In [ ]:
summary = pd.DataFrame(SUMMARY)
summary.to_csv(EXPORT_DIR / "export_summary.csv", index=False)
display(summary.drop(columns=["per_class"]))

bad = summary[(summary.variant == "fp32") & ((summary.agree_torch < 99.5) | (summary.acc_drop_vs_torch.abs() > 0.2))]
if len(bad):
    print("CẢNH BÁO: bản FP32 không tương đương PyTorch ở:", bad[["dataset", "agree_torch", "acc_drop_vs_torch"]].to_dict("records"))
else:
    print("OK: mọi bản FP32 tương đương PyTorch.")

print("\nĐộ chính xác từng lớp (%):")
for _, r in summary.iterrows():
    print(f"  {r.dataset:8s} {r.variant:12s}", json.loads(r.per_class))

## 9. Đóng gói thành .ferpack cho fer-edge
Mỗi biến thể thành **một file `.ferpack`** (zip gồm `fer.json` + `model.onnx` + `testset/`). Đây là thứ duy nhất cần mang sang máy khác: tải ở tab **Output**, rồi đưa lên **GitHub Releases** để `fer-edge` tự tải về.

In [ ]:
import hashlib

ARCH_LABEL = {"resnet18": "ResNet-18", "apvit": "APViT", "vit_b16": "ViT-B/16",
              "posterv2": "POSTER V2", "clip_lora": "CLIP ViT-B/16 + LoRA"}
VARIANT_SUFFIX = {"fp32": "fp32", "int8_dyn": "int8d", "int8_static": "int8s"}
PACK_DIR = Path("/kaggle/working/packs")
PACK_DIR.mkdir(parents=True, exist_ok=True)
entries = {}

for name in DATASETS_TO_RUN:
    d = EXPORT_DIR / name
    meta_all = json.loads((d / "meta.json").read_text(encoding="utf-8"))
    te = meta_all.get("test_eval", {})
    base = {k: v for k, v in meta_all.items()
            if k not in ("files", "test_eval", "quant_errors", "pi_subset", "results_csv_ref")}
    for variant, fname in meta_all["files"].items():
        onnx_file = d / fname
        if not onnx_file.is_file():
            continue
        keep = ("acc", "mca", "f1", "per_class")
        meta = {**base, "name": f"{MODEL_NAME}-{name}-{VARIANT_SUFFIX.get(variant, variant)}",
                "variant": variant, "onnx": "model.onnx", "arch": ARCH_LABEL.get(MODEL_NAME, MODEL_NAME),
                "metrics": {k: v for k, v in (te.get(variant) or {}).items() if k in keep},
                "metrics_torch_fp32": {k: v for k, v in (te.get("torch_fp32") or {}).items() if k in keep},
                "n_test": te.get("n"), "agree_torch": (te.get(variant) or {}).get("agree_torch"),
                "max_abs_logit_diff": (te.get(variant) or {}).get("max_abs_logit_diff")}
        out = PACK_DIR / f"{meta['name']}.ferpack"
        with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
            z.writestr("fer.json", json.dumps(meta, indent=1, ensure_ascii=False))
            z.write(onnx_file, "model.onnx")
            sub = d / "pi_subset"
            for f in sorted(sub.rglob("*")) if sub.is_dir() else []:
                if f.is_file():
                    z.write(f, f"testset/{f.relative_to(sub).as_posix()}")
        h = hashlib.sha256(out.read_bytes()).hexdigest()
        entries[meta["name"]] = {"url": f"<URL release>/{out.name}", "sha256": h,
                                 "size_mb": round(out.stat().st_size / 1e6, 1)}
        print(f"  {out.name:34s} {out.stat().st_size / 1e6:7.1f} MB")

print("\nMục cho src/fer_edge/models.json (thay <URL release> bằng link GitHub Release):")
print(json.dumps(entries, indent=2))
print(f"\nTải {len(entries)} file trong {PACK_DIR} ở tab Output.")

## Ghi chú cho bài báo
- Export ONNX opset 17 (bộ export TorchScript của PyTorch), batch 1, đầu vào float32 `1×3×S×S`; suy luận bằng ONNX Runtime trên CPU.
- INT8 động: `quantize_dynamic` (trọng số INT8 theo tensor). INT8 tĩnh: `quantize_static`, định dạng QDQ, trọng số INT8 theo kênh, activation UINT8, hiệu chỉnh MinMax trên 200 ảnh train ngẫu nhiên (seed 0) qua tiền xử lý đánh giá.
- Báo cáo độ chính xác của mỗi bản trên **toàn bộ tập test** (bảng `export_summary.csv`) và tỉ lệ dự đoán trùng với PyTorch FP32.
- Độ chính xác trong `results.csv` được đo với AMP (FP16) lúc huấn luyện; `torch_fp32` ở đây đo FP32 nên có thể lệch vài phần trăm của một điểm.